In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import joblib

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

from IPython.display import display

print("Feature Engineering Notebook Started")

Feature Engineering Notebook Started


In [2]:
# Project paths
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

# Create required folders
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nFeatures directory:")
print(FEATURES_DIR)

print("\nModels directory:")
print(MODELS_DIR)

Project directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation

Processed directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\processed

Features directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features

Models directory:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\models


In [3]:
print("Checking processed files...\n")

for file in PROCESSED_DIR.iterdir():
    print(file.name)

Checking processed files...

destinations_clean.csv
evaluation_cases_clean.csv
preprocessing_report.csv
user_profiles_clean.csv


In [4]:
destinations = pd.read_csv(
    PROCESSED_DIR / "destinations_clean.csv"
)

users = pd.read_csv(
    PROCESSED_DIR / "user_profiles_clean.csv"
)

evaluation_cases = pd.read_csv(
    PROCESSED_DIR / "evaluation_cases_clean.csv"
)

print("Datasets loaded successfully.\n")

print("Destinations:", destinations.shape)
print("Users:", users.shape)
print("Evaluation cases:", evaluation_cases.shape)

Datasets loaded successfully.

Destinations: (100, 21)
Users: (50, 13)
Evaluation cases: (500, 27)


In [5]:
print("========== DESTINATION COLUMNS ==========")
print(destinations.columns.tolist())

print("\n========== USER COLUMNS ==========")
print(users.columns.tolist())

print("\n========== EVALUATION COLUMNS ==========")
print(evaluation_cases.columns.tolist())

========== DESTINATION COLUMNS ==========
['destination_id', 'name', 'district', 'latitude', 'longitude', 'category', 'estimated_cost_lkr', 'duration_hours', 'interest_tags', 'sustainability_score', 'crowd_score', 'seasonality_score', 'sentiment_nature', 'sentiment_service', 'sentiment_value', 'weather_temp_c', 'weather_rain_risk', 'weather_suitability', 'source_provenance', 'data_status', 'num_interest_tags']

========== USER COLUMNS ==========
['user_id', 'age_group', 'traveller_type', 'travel_style', 'budget_lkr_per_day', 'trip_duration_days', 'starting_location', 'preferred_interests', 'preferred_crowd', 'preferred_weather', 'sustainability_importance', 'data_status', 'num_preferred_interests']

========== EVALUATION COLUMNS ==========
['case_id', 'user_id', 'destination_id', 'scenario_id', 'user_budget_lkr_per_day', 'trip_duration_days', 'travel_style', 'starting_location', 'preferred_interests', 'preferred_crowd', 'preferred_weather', 'destination_name', 'destination_category', '

In [6]:
destination_numeric_columns = [
    "latitude",
    "longitude",
    "estimated_cost_lkr",
    "duration_hours",
    "sustainability_score",
    "crowd_score",
    "seasonality_score",
    "sentiment_nature",
    "sentiment_service",
    "sentiment_value",
    "weather_temp_c",
    "weather_rain_risk",
    "weather_suitability",
    "num_interest_tags"
]

user_numeric_columns = [
    "budget_lkr_per_day",
    "trip_duration_days",
    "sustainability_importance",
    "num_preferred_interests"
]

evaluation_numeric_columns = [
    "user_budget_lkr_per_day",
    "trip_duration_days",
    "estimated_cost_lkr",
    "duration_hours",
    "distance_km",
    "preference_score",
    "budget_score",
    "sustainability_score",
    "crowd_fit_score",
    "context_score",
    "distance_score",
    "sentiment_score",
    "popularity_score",
    "overall_hybrid_score",
    "relevance_rating_1_to_5"
]

for col in destination_numeric_columns:
    if col in destinations.columns:
        destinations[col] = pd.to_numeric(
            destinations[col],
            errors="coerce"
        )

for col in user_numeric_columns:
    if col in users.columns:
        users[col] = pd.to_numeric(
            users[col],
            errors="coerce"
        )

for col in evaluation_numeric_columns:
    if col in evaluation_cases.columns:
        evaluation_cases[col] = pd.to_numeric(
            evaluation_cases[col],
            errors="coerce"
        )

print("Numeric conversion completed.")

Numeric conversion completed.


In [7]:
for col in destination_numeric_columns:
    if col in destinations.columns:
        destinations[col] = destinations[col].fillna(
            destinations[col].median()
        )

for col in user_numeric_columns:
    if col in users.columns:
        users[col] = users[col].fillna(
            users[col].median()
        )

for col in evaluation_numeric_columns:
    if col in evaluation_cases.columns:
        evaluation_cases[col] = evaluation_cases[col].fillna(
            evaluation_cases[col].median()
        )

print("Missing numeric values handled.")

Missing numeric values handled.


In [8]:
def clean_text(value):
    if pd.isna(value):
        return ""
    
    value = str(value).lower().strip()
    
    # Replace separators with spaces
    value = value.replace(";", " ")
    value = value.replace(",", " ")
    value = value.replace("|", " ")
    
    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)
    
    return value

In [9]:
destinations["destination_text"] = (
    destinations["name"].apply(clean_text)
    + " "
    + destinations["district"].apply(clean_text)
    + " "
    + destinations["category"].apply(clean_text)
    + " "
    + destinations["interest_tags"].apply(clean_text)
)

print("Destination text created.")

display(
    destinations[
        [
            "destination_id",
            "name",
            "category",
            "interest_tags",
            "destination_text"
        ]
    ].head(10)
)

Destination text created.


,destination_id,name,category,interest_tags,destination_text
0,D001,Sigiriya Rock Fortress,heritage;nature,heritage;history;photography;hiking,sigiriya rock fortress matale heritage nature ...
1,D002,Dambulla Cave Temple,heritage,heritage;history;culture;photography,dambulla cave temple matale heritage heritage ...
2,D003,Minneriya National Park,wildlife;nature,wildlife;photography;nature;adventure,minneriya national park polonnaruwa wildlife n...
3,D004,Kaudulla National Park,wildlife;nature,wildlife;photography;nature;adventure,kaudulla national park polonnaruwa wildlife na...
4,D005,Polonnaruwa Ancient City,heritage,heritage;history;culture;photography,polonnaruwa ancient city polonnaruwa heritage ...
5,D006,Anuradhapura Sacred City,heritage;religious,heritage;history;culture;religious;photography,anuradhapura sacred city anuradhapura heritage...
6,D007,Ritigala Monastery,heritage;nature,heritage;history;nature;hiking,ritigala monastery anuradhapura heritage natur...
7,D008,Mihintale,heritage;religious,heritage;history;religious;photography,mihintale anuradhapura heritage religious heri...
8,D009,Wilpattu National Park,wildlife;nature,wildlife;nature;photography;adventure,wilpattu national park puttalam wildlife natur...
9,D010,Jaffna Fort,heritage,heritage;history;culture;photography,jaffna fort jaffna heritage heritage history c...


In [10]:
users["user_preference_text"] = (
    users["preferred_interests"].apply(clean_text)
    + " "
    + users["travel_style"].apply(clean_text)
    + " "
    + users["traveller_type"].apply(clean_text)
    + " "
    + users["preferred_crowd"].apply(clean_text)
    + " "
    + users["preferred_weather"].apply(clean_text)
    + " "
    + users["starting_location"].apply(clean_text)
)

print("User preference text created.")

display(
    users[
        [
            "user_id",
            "preferred_interests",
            "travel_style",
            "traveller_type",
            "user_preference_text"
        ]
    ].head(10)
)

User preference text created.


,user_id,preferred_interests,travel_style,traveller_type,user_preference_text
0,U001,nature;photography,budget,solo,nature photography budget solo low cool colombo
1,U002,beach;relaxation,moderate,couple,beach relaxation moderate couple medium warm k...
2,U003,heritage;history,budget,friends,heritage history budget friends low dry gampaha
3,U004,wildlife;photography,moderate,family,wildlife photography moderate family medium mi...
4,U005,adventure;hiking,premium,solo,adventure hiking premium solo high rain_tolera...
5,U006,culture;food,budget,couple,culture food budget couple low cool galle
6,U007,nature;hiking,moderate,friends,nature hiking moderate friends medium warm kur...
7,U008,beach;adventure,budget,family,beach adventure budget family low dry ratnapura
8,U009,heritage;culture,moderate,solo,heritage culture moderate solo medium mixed ba...
9,U010,wildlife;nature,premium,couple,wildlife nature premium couple high rain_toler...


In [11]:
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words="english"
)

destination_tfidf_matrix = tfidf.fit_transform(
    destinations["destination_text"]
)

print("TF-IDF matrix created.")
print("Shape:", destination_tfidf_matrix.shape)

TF-IDF matrix created.
Shape: (100, 500)


In [12]:
tfidf_path = FEATURES_DIR / "destination_tfidf_vectorizer.joblib"

joblib.dump(
    tfidf,
    tfidf_path
)

print("TF-IDF vectorizer saved:")
print(tfidf_path)

TF-IDF vectorizer saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\destination_tfidf_vectorizer.joblib


In [13]:
destination_feature_columns = [
    "estimated_cost_lkr",
    "duration_hours",
    "sustainability_score",
    "crowd_score",
    "seasonality_score",
    "sentiment_nature",
    "sentiment_service",
    "sentiment_value",
    "weather_temp_c",
    "weather_rain_risk",
    "weather_suitability",
    "num_interest_tags"
]

destination_features = destinations[
    ["destination_id"] + destination_feature_columns
].copy()

display(destination_features.head())

,destination_id,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,sentiment_nature,sentiment_service,sentiment_value,weather_temp_c,weather_rain_risk,weather_suitability,num_interest_tags
0,D001,12000,3.0,57,41,58,0.699,0.774,0.675,25.1,0.484,0.516,1
1,D002,5000,2.0,55,41,47,0.727,0.864,0.910,28.9,0.246,0.754,1
2,D003,15000,4.0,70,35,59,0.745,0.866,0.705,24.1,0.618,0.382,1
3,D004,15000,4.0,64,33,66,0.619,0.639,0.610,28.4,0.634,0.366,1
4,D005,5000,4.0,58,32,62,0.785,0.943,0.652,27.1,0.533,0.467,1


In [14]:
destination_scaler = MinMaxScaler()

destination_features_scaled = destination_scaler.fit_transform(
    destination_features[destination_feature_columns]
)

destination_features_scaled = pd.DataFrame(
    destination_features_scaled,
    columns=[
        f"{col}_normalized"
        for col in destination_feature_columns
    ]
)

destination_features_final = pd.concat(
    [
        destination_features[["destination_id"]].reset_index(drop=True),
        destination_features_scaled.reset_index(drop=True)
    ],
    axis=1
)

print("Destination features normalized.")

display(destination_features_final.head())

Destination features normalized.


,destination_id,estimated_cost_lkr_normalized,duration_hours_normalized,sustainability_score_normalized,crowd_score_normalized,seasonality_score_normalized,sentiment_nature_normalized,sentiment_service_normalized,sentiment_value_normalized,weather_temp_c_normalized,weather_rain_risk_normalized,weather_suitability_normalized,num_interest_tags_normalized
0,D001,0.714286,0.166667,0.285714,0.833333,0.523810,0.257396,0.567089,0.299742,0.147059,0.751174,0.248826,0.0
1,D002,0.214286,0.000000,0.190476,0.833333,0.000000,0.340237,0.794937,0.906977,0.705882,0.378717,0.621283,0.0
2,D003,0.928571,0.333333,0.904762,0.583333,0.571429,0.393491,0.800000,0.377261,0.000000,0.960876,0.039124,0.0
3,D004,0.928571,0.333333,0.619048,0.500000,0.904762,0.020710,0.225316,0.131783,0.632353,0.985915,0.014085,0.0
4,D005,0.214286,0.333333,0.333333,0.458333,0.714286,0.511834,0.994937,0.240310,0.441176,0.827856,0.172144,0.0


In [15]:
destination_scaler_path = (
    FEATURES_DIR / "destination_scaler.joblib"
)

joblib.dump(
    destination_scaler,
    destination_scaler_path
)

print("Destination scaler saved:")
print(destination_scaler_path)

Destination scaler saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\destination_scaler.joblib


In [16]:
user_feature_columns = [
    "budget_lkr_per_day",
    "trip_duration_days",
    "sustainability_importance",
    "num_preferred_interests"
]

user_features = users[
    ["user_id"] + user_feature_columns
].copy()

display(user_features.head())

,user_id,budget_lkr_per_day,trip_duration_days,sustainability_importance,num_preferred_interests
0,U001,5000,1,0.40,1
1,U002,8000,2,0.48,1
2,U003,11000,3,0.56,1
3,U004,14000,4,0.64,1
4,U005,17000,5,0.72,1


In [17]:
user_scaler = MinMaxScaler()

user_features_scaled = user_scaler.fit_transform(
    user_features[user_feature_columns]
)

user_features_scaled = pd.DataFrame(
    user_features_scaled,
    columns=[
        f"{col}_normalized"
        for col in user_feature_columns
    ]
)

user_features_final = pd.concat(
    [
        user_features[["user_id"]].reset_index(drop=True),
        user_features_scaled.reset_index(drop=True)
    ],
    axis=1
)

print("User features normalized.")

display(user_features_final.head())

User features normalized.


,user_id,budget_lkr_per_day_normalized,trip_duration_days_normalized,sustainability_importance_normalized,num_preferred_interests_normalized
0,U001,0.0,0.00,0.000000,0.0
1,U002,0.2,0.25,0.166667,0.0
2,U003,0.4,0.50,0.333333,0.0
3,U004,0.6,0.75,0.500000,0.0
4,U005,0.8,1.00,0.666667,0.0


In [18]:
user_scaler_path = FEATURES_DIR / "user_scaler.joblib"

joblib.dump(
    user_scaler,
    user_scaler_path
)

print("User scaler saved:")
print(user_scaler_path)

User scaler saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\user_scaler.joblib


In [19]:
def split_tags(value):
    if pd.isna(value):
        return []
    
    return [
        item.strip().lower()
        for item in str(value).split(";")
        if item.strip()
    ]


destination_categories = destinations[
    "category"
].apply(split_tags)

category_encoder = MultiLabelBinarizer()

destination_category_matrix = category_encoder.fit_transform(
    destination_categories
)

destination_category_df = pd.DataFrame(
    destination_category_matrix,
    columns=[
        f"category_{c}"
        for c in category_encoder.classes_
    ]
)

print("Destination category features created.")

display(destination_category_df.head())

Destination category features created.


,category_adventure,category_beach,category_city,category_coastal,category_culture,category_family,category_heritage,category_marine,category_nature,category_religious,category_village,category_wildlife
0,0,0,0,0,0,0,1,0,1,0,0,0
1,0,0,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,0,1
3,0,0,0,0,0,0,0,0,1,0,0,1
4,0,0,0,0,0,0,1,0,0,0,0,0


In [20]:
category_encoder_path = (
    FEATURES_DIR / "destination_category_encoder.joblib"
)

joblib.dump(
    category_encoder,
    category_encoder_path
)

print("Category encoder saved.")

Category encoder saved.


In [21]:
destination_interests = destinations[
    "interest_tags"
].apply(split_tags)

interest_encoder = MultiLabelBinarizer()

destination_interest_matrix = interest_encoder.fit_transform(
    destination_interests
)

destination_interest_df = pd.DataFrame(
    destination_interest_matrix,
    columns=[
        f"interest_{c}"
        for c in interest_encoder.classes_
    ]
)

print("Destination interest features created.")

display(destination_interest_df.head())

Destination interest features created.


,interest_adventure,interest_beach,interest_birdwatching,interest_boat_trip,interest_culture,interest_education,interest_family,interest_food,interest_heritage,interest_hiking,...,interest_nature,interest_photography,interest_rafting,interest_relaxation,interest_religious,interest_shopping,interest_snorkeling,interest_surfing,interest_water_sports,interest_wildlife
0,0,0,0,0,0,0,0,0,1,1,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,1
3,1,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,1
4,0,0,0,0,1,0,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0


In [22]:
interest_encoder_path = (
    FEATURES_DIR / "destination_interest_encoder.joblib"
)

joblib.dump(
    interest_encoder,
    interest_encoder_path
)

print("Interest encoder saved.")

Interest encoder saved.


In [23]:
destination_feature_dataset = pd.concat(
    [
        destinations[
            [
                "destination_id",
                "name",
                "district",
                "destination_text"
            ]
        ].reset_index(drop=True),

        destination_features_final.drop(
            columns=["destination_id"]
        ).reset_index(drop=True),

        destination_category_df.reset_index(drop=True),

        destination_interest_df.reset_index(drop=True)
    ],
    axis=1
)

print(
    "Final destination feature dataset:",
    destination_feature_dataset.shape
)

display(destination_feature_dataset.head())

Final destination feature dataset: (100, 50)


,destination_id,name,district,destination_text,estimated_cost_lkr_normalized,duration_hours_normalized,sustainability_score_normalized,crowd_score_normalized,seasonality_score_normalized,sentiment_nature_normalized,...,interest_nature,interest_photography,interest_rafting,interest_relaxation,interest_religious,interest_shopping,interest_snorkeling,interest_surfing,interest_water_sports,interest_wildlife
0,D001,Sigiriya Rock Fortress,matale,sigiriya rock fortress matale heritage nature ...,0.714286,0.166667,0.285714,0.833333,0.523810,0.257396,...,0,1,0,0,0,0,0,0,0,0
1,D002,Dambulla Cave Temple,matale,dambulla cave temple matale heritage heritage ...,0.214286,0.000000,0.190476,0.833333,0.000000,0.340237,...,0,1,0,0,0,0,0,0,0,0
2,D003,Minneriya National Park,polonnaruwa,minneriya national park polonnaruwa wildlife n...,0.928571,0.333333,0.904762,0.583333,0.571429,0.393491,...,1,1,0,0,0,0,0,0,0,1
3,D004,Kaudulla National Park,polonnaruwa,kaudulla national park polonnaruwa wildlife na...,0.928571,0.333333,0.619048,0.500000,0.904762,0.020710,...,1,1,0,0,0,0,0,0,0,1
4,D005,Polonnaruwa Ancient City,polonnaruwa,polonnaruwa ancient city polonnaruwa heritage ...,0.214286,0.333333,0.333333,0.458333,0.714286,0.511834,...,0,1,0,0,0,0,0,0,0,0


In [24]:
destination_features_path = (
    FEATURES_DIR / "destination_features.csv"
)

destination_feature_dataset.to_csv(
    destination_features_path,
    index=False
)

print("Saved:")
print(destination_features_path)

Saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\destination_features.csv


In [25]:
user_feature_dataset = pd.concat(
    [
        users[
            [
                "user_id",
                "age_group",
                "traveller_type",
                "travel_style",
                "starting_location",
                "preferred_interests",
                "preferred_crowd",
                "preferred_weather",
                "user_preference_text"
            ]
        ].reset_index(drop=True),

        user_features_final.drop(
            columns=["user_id"]
        ).reset_index(drop=True)
    ],
    axis=1
)

print(
    "Final user feature dataset:",
    user_feature_dataset.shape
)

display(user_feature_dataset.head())

Final user feature dataset: (50, 13)


,user_id,age_group,traveller_type,travel_style,starting_location,preferred_interests,preferred_crowd,preferred_weather,user_preference_text,budget_lkr_per_day_normalized,trip_duration_days_normalized,sustainability_importance_normalized,num_preferred_interests_normalized
0,U001,18-24,solo,budget,colombo,nature;photography,low,cool,nature photography budget solo low cool colombo,0.0,0.00,0.000000,0.0
1,U002,25-34,couple,moderate,kandy,beach;relaxation,medium,warm,beach relaxation moderate couple medium warm k...,0.2,0.25,0.166667,0.0
2,U003,35-44,friends,budget,gampaha,heritage;history,low,dry,heritage history budget friends low dry gampaha,0.4,0.50,0.333333,0.0
3,U004,18-24,family,moderate,negombo,wildlife;photography,medium,mixed,wildlife photography moderate family medium mi...,0.6,0.75,0.500000,0.0
4,U005,25-34,solo,premium,matara,adventure;hiking,high,rain_tolerant,adventure hiking premium solo high rain_tolera...,0.8,1.00,0.666667,0.0


In [26]:
user_features_path = (
    FEATURES_DIR / "user_features.csv"
)

user_feature_dataset.to_csv(
    user_features_path,
    index=False
)

print("Saved:")
print(user_features_path)

Saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\user_features.csv


In [27]:
evaluation_feature_columns = [
    "case_id",
    "user_id",
    "destination_id",
    "scenario_id",
    "user_budget_lkr_per_day",
    "trip_duration_days",
    "distance_km",
    "preference_score",
    "budget_score",
    "sustainability_score",
    "crowd_fit_score",
    "context_score",
    "distance_score",
    "sentiment_score",
    "popularity_score",
    "overall_hybrid_score",
    "relevance_rating_1_to_5"
]

available_evaluation_columns = [
    col
    for col in evaluation_feature_columns
    if col in evaluation_cases.columns
]

evaluation_features = evaluation_cases[
    available_evaluation_columns
].copy()

print(
    "Evaluation features:",
    evaluation_features.shape
)

display(evaluation_features.head())

Evaluation features: (500, 17)


,case_id,user_id,destination_id,scenario_id,user_budget_lkr_per_day,trip_duration_days,distance_km,preference_score,budget_score,sustainability_score,crowd_fit_score,context_score,distance_score,sentiment_score,popularity_score,overall_hybrid_score,relevance_rating_1_to_5
0,C001,U001,D001,1,5000,1,151.47,0.5,0.0,0.57,0.84,0.516,0.621,0.716,0.41,0.505,3
1,C002,U001,D012,2,5000,1,319.61,0.5,0.8,0.58,0.82,0.866,0.201,0.814,0.43,0.627,4
2,C003,U001,D023,3,5000,1,104.95,1.0,0.6,0.63,0.95,0.461,0.738,0.814,0.30,0.781,4
3,C004,U001,D034,4,5000,1,128.57,1.0,0.8,0.67,0.88,0.520,0.679,0.825,0.37,0.811,4
4,C005,U001,D045,5,5000,1,237.86,0.5,0.6,0.57,0.83,0.520,0.405,0.751,0.42,0.576,3


In [28]:
evaluation_features_path = (
    FEATURES_DIR / "evaluation_features.csv"
)

evaluation_features.to_csv(
    evaluation_features_path,
    index=False
)

print("Saved:")
print(evaluation_features_path)

Saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features\evaluation_features.csv


In [29]:
destination_text_dataset = destinations[
    [
        "destination_id",
        "name",
        "destination_text"
    ]
].copy()

user_text_dataset = users[
    [
        "user_id",
        "user_preference_text"
    ]
].copy()

destination_text_dataset.to_csv(
    FEATURES_DIR / "destination_text.csv",
    index=False
)

user_text_dataset.to_csv(
    FEATURES_DIR / "user_text.csv",
    index=False
)

print("Text feature datasets saved.")

Text feature datasets saved.


In [30]:
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)

print("\nDestination feature dataset:")
print(destination_feature_dataset.shape)

print("\nUser feature dataset:")
print(user_feature_dataset.shape)

print("\nEvaluation feature dataset:")
print(evaluation_features.shape)

print("\nFeature files:")
for file in sorted(FEATURES_DIR.iterdir()):
    print(" -", file.name)

print("\nModel files:")
for file in sorted(MODELS_DIR.iterdir()):
    print(" -", file.name)

print("\nAll feature engineering steps completed successfully.")

FEATURE ENGINEERING COMPLETE

Destination feature dataset:
(100, 50)

User feature dataset:
(50, 13)

Evaluation feature dataset:
(500, 17)

Feature files:
 - destination_category_encoder.joblib
 - destination_features.csv
 - destination_interest_encoder.joblib
 - destination_sbert_embeddings.npy
 - destination_scaler.joblib
 - destination_text.csv
 - destination_tfidf_vectorizer.joblib
 - destinations_with_crowd_predictions.csv
 - evaluation_features.csv
 - user_features.csv
 - user_sbert_embeddings.npy
 - user_scaler.joblib
 - user_text.csv

Model files:
 - crowd_prediction_random_forest.joblib
 - sbert_model.joblib

All feature engineering steps completed successfully.
